# Starter Notebook: Qwen 2B LoRA for Text-to-SVG (Kaggle)

This starter is built from the resources in `contest_docs`:
- Data resources: `contest_docs/03_Data_Design.md`
- Baseline and starter guidance: `contest_docs/05_Baselines_and_Starter_Notebooks.md`
- Kaggle implementation notes: `contest_docs/06_Kaggle_Implementation_Guide.md`

Goal: provide a practical scaffold for Qwen-2B-class fine-tuning + submission generation.

## Referenced Data and Docs

### Dataset resources from `contest_docs/03_Data_Design.md`
- `OmniSVG/MMSVG-Icon`
- `xingxm/SVGX-Core-250k`
- `xingxm/SVGX-SFT-1M` (recommended subset: `SVGX_SFT_GEN_51k.json`)
- `nyuuzyou/svgfind`
- `starvector/svg-icons`
- `thesantatitan/deepseek-svg-dataset`
- `InternSVG/SArena` (evaluation benchmark)

### Qwen 2B fine-tuning references from `contest_docs/05` and `contest_docs/06`
- Unsloth Qwen fine-tune docs: https://unsloth.ai/docs/models/qwen3.5/fine-tune
- Qwen3.5-2B Vision notebook: https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_(2B)_Vision.ipynb

Note: this notebook is written as a reusable starter. You may need to adjust exact model IDs and column names to match the latest upstream datasets.

In [1]:
# Uncomment in a fresh Kaggle notebook environment.
%pip install -q unsloth datasets trl transformers accelerate peft bitsandbytes pandas lxml vllm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.8/54.8 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.2/433.2 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 156.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from huggingface_hub import login
# Paste your copied token below
login(token="hf_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")

In [ ]:
import os
import re
import time
import random
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch

from datasets import concatenate_datasets, load_dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Torch: 2.10.0+cu128
CUDA available: True


In [ ]:
# Core training config.
# Keep runtime targets in line with contest_docs guidance (roughly <= 6-8 hours training).
CONFIG = {
    "model_name": "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit",
    "max_seq_length": 3072,  # Bumped from 2048 to prevent 8,000-char SVGs from getting cut off
    "lora_r": 32,
    "lora_alpha": 64,
    "learning_rate": 3e-4,
    "num_train_epochs": 4,
    "per_device_train_batch_size": 16,  # Pushed from 2 to 16. The H100 has 80GB VRAM, use it!
    "gradient_accumulation_steps": 2,   # Dropped from 8 since our batch size is so large

    "warmup_ratio": 0.1,
    "weight_decay": 0.01,
    "logging_steps": 20,
    "eval_steps": 100,
    "save_steps": 200,

    # 4. FULL DATASET
    "max_train_samples_per_source": 50000, # H100 is fast enough to train on the whole 50k dataset in a few hours

    "eval_size": 0.02,
    "output_dir": "/kaggle/working/qwen2b_svg_lora",
}

CONFIG

{'model_name': 'unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit',
 'max_seq_length': 3072,
 'lora_r': 32,
 'lora_alpha': 64,
 'learning_rate': 0.0003,
 'num_train_epochs': 4,
 'per_device_train_batch_size': 16,
 'gradient_accumulation_steps': 2,
 'warmup_ratio': 0.1,
 'weight_decay': 0.01,
 'logging_steps': 20,
 'eval_steps': 100,
 'save_steps': 200,
 'max_train_samples_per_source': 50000,
 'eval_size': 0.02,
 'output_dir': '/kaggle/working/qwen2b_svg_lora'}

In [ ]:
# Data catalog using the resources listed in contest_docs/03_Data_Design.md.
DATASET_CATALOG = {
    "OmniSVG/MMSVG-Icon": {
        "split": "train",
        "prompt_fields": ["description", "keywords", "detail", "prompt", "text"],
        "svg_fields": ["svg", "picosvg", "completion", "target"],
    },
    "xingxm/SVGX-Core-250k": {
        "split": "train",
        "prompt_fields": ["qwen_caption", "blip_caption", "name", "img_analysis", "prompt"],
        "svg_fields": ["svg", "completion", "target"],
    },
    "xingxm/SVGX-SFT-1M": {
        "split": "train",
        "prompt_fields": ["prompt", "instruction", "input", "query"],
        "svg_fields": ["completion", "output", "svg", "response"],
    },
    "thesantatitan/deepseek-svg-dataset": {
        "split": "train",
        "prompt_fields": ["prompt", "instruction", "input"],
        "svg_fields": ["completion", "output", "svg"],
    },
}

# For a first run, keep to 1-2 sources.
ACTIVE_SOURCES = [
    "xingxm/SVGX-SFT-1M",
    "OmniSVG/MMSVG-Icon",
]

In [ ]:
def _pick_first_non_empty(example, keys):
    for key in keys:
        if key in example and example[key] is not None:
            val = str(example[key]).strip()
            if val:
                return val
    return ""


def to_prompt_svg(example, prompt_fields, svg_fields):
    prompt = _pick_first_non_empty(example, prompt_fields)
    svg = _pick_first_non_empty(example, svg_fields)
    if not svg.lower().startswith("<svg"):
        return {"prompt": "", "svg": ""}
    return {"prompt": prompt, "svg": svg}


def load_source_dataset(dataset_id, cfg, max_samples):
    print(f"Loading {dataset_id} ...")
    ds = load_dataset(dataset_id, split=cfg["split"])
    if max_samples and len(ds) > max_samples:
        ds = ds.shuffle(seed=SEED).select(range(max_samples))
    ds = ds.map(
        lambda ex: to_prompt_svg(ex, cfg["prompt_fields"], cfg["svg_fields"]),
        remove_columns=ds.column_names,
        desc=f"normalizing {dataset_id}",
    )
    ds = ds.filter(lambda x: bool(x["prompt"]) and bool(x["svg"]))
    print(f"{dataset_id}: {len(ds)} usable rows")
    return ds

In [ ]:
datasets_ok = []
for source in ACTIVE_SOURCES:
    try:
        ds = load_source_dataset(
            source,
            DATASET_CATALOG[source],
            CONFIG["max_train_samples_per_source"],
        )
        datasets_ok.append(ds)
    except Exception as e:
        print(f"Skipping {source}: {type(e).__name__}: {e}")

if not datasets_ok:
    raise RuntimeError("No dataset loaded. Check dataset IDs, internet access, and schema fields.")

train_raw = datasets_ok[0] if len(datasets_ok) == 1 else concatenate_datasets(datasets_ok)
train_raw = train_raw.shuffle(seed=SEED)

splits = train_raw.train_test_split(test_size=CONFIG["eval_size"], seed=SEED)
train_ds = splits["train"]
eval_ds = splits["test"]

print(f"Train rows: {len(train_ds)}")
print(f"Eval rows: {len(eval_ds)}")
train_ds[0]

Loading xingxm/SVGX-SFT-1M ...


README.md: 0.00B [00:00, ?B/s]

SVGX_SFT_GEN_51k.json:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

SVGX_SFT_GEN_51k_encode.json:   0%|          | 0.00/1.56G [00:00<?, ?B/s]

SVGX_SFT_GEN_51k_int.json:   0%|          | 0.00/754M [00:00<?, ?B/s]

...

Loading dataset shards:   0%|          | 0/38 [00:00<?, ?it/s]

normalizing OmniSVG/MMSVG-Icon:   0%|          | 0/50000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/50000 [00:00<?, ? examples/s]

OmniSVG/MMSVG-Icon: 50000 usable rows
Train rows: 49000
Eval rows: 1000


{'svg': '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0.0 0.0 200.0 200.0" height="200.0px" width="200.0px"><path fill="#F69661" fill-opacity="1.0"  filling="0" d="M185.71400451660156 7.142997741699219 L14.286003112792969 7.142997741699219 C10.337997436523438 7.142997741699219 7.142997741699219 10.34100341796875 7.142997741699219 14.286003112792969 L7.142997741699219 185.71400451660156 C7.142997741699219 189.66299438476562 10.337997436523438 192.85699462890625 14.286003112792969 192.85699462890625 L185.71400451660156 192.85699462890625 C189.66299438476562 192.85699462890625 192.85699462890625 189.66299438476562 192.85699462890625 185.71400451660156 L192.85699462890625 14.286003112792969 C192.85699462890625 10.337997436523438 189.66299438476562 7.142997741699219 185.71400451660156 7.142997741699219 Z M173.52099609375 21.429000854492188 L142.85699462890625 52.09299850463867 L112.19300079345703 21.429000854492188 L173.52099609375 21.429000854492188 Z M137.8070068359375 57.143001556396

In [ ]:
SYSTEM_PROMPT = (
    "You generate compact, valid SVG markup from user requests. "
    "Return only SVG code with a single root <svg> element."
)


def format_sft_text(example):
    text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{example['prompt']}<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{example['svg']}<|im_end|>"
    )
    return {"text": text}


train_text = train_ds.map(format_sft_text, remove_columns=train_ds.column_names)
eval_text = eval_ds.map(format_sft_text, remove_columns=eval_ds.column_names)

print(train_text[0]["text"][:400])

Map:   0%|          | 0/49000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

<|im_start|>system
You generate compact, valid SVG markup from user requests. Return only SVG code with a single root <svg> element.<|im_end|>
<|im_start|>user
An orange square with intersecting lines forming a geometric pattern.<|im_end|>
<|im_start|>assistant
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0.0 0.0 200.0 200.0" height="200.0px" width="200.0px"><path fill="#F69661" fill-opacity="


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.15: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.179 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/265 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

Unsloth 2026.3.15 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_ratio=CONFIG["warmup_ratio"],
    weight_decay=CONFIG["weight_decay"],
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=CONFIG["logging_steps"],
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=2,
    report_to="none",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_text,
    eval_dataset=eval_text,
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    packing=True,
    args=training_args,
)

train_result = trainer.train()
train_result

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=30):   0%|          | 0/49000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=30):   0%|          | 0/1000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 49,000 | Num Epochs = 4 | Total steps = 6,128
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 2 x 1) = 32
 "-____-"     Trainable parameters = 36,929,536 of 1,580,643,840 (2.34% trained)


Step,Training Loss,Validation Loss
100,0.451411,0.438695
200,0.381819,0.376395
300,0.354787,0.352775
400,0.353887,0.337219
500,0.333184,0.322138
600,0.317450,0.312906
700,0.307219,0.305601
800,0.302918,0.299853
900,0.301361,0.295421
1000,0.298940,0.291003


Unsloth: Not an error, but Qwen2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transform

TrainOutput(global_step=6128, training_loss=0.2665017451069685, metrics={'train_runtime': 5094.5086, 'train_samples_per_second': 38.473, 'train_steps_per_second': 1.203, 'total_flos': 1.622415041888256e+18, 'train_loss': 0.2665017451069685, 'epoch': 4.0})

In [ ]:
os.makedirs(CONFIG["output_dir"], exist_ok=True)
trainer.save_model(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])

print(f"Saved adapter + tokenizer to: {CONFIG['output_dir']}")

Saved adapter + tokenizer to: /kaggle/working/qwen2b_svg_lora


In [ ]:
FastLanguageModel.for_inference(model)

SVG_REGEX = re.compile(r"<svg[\s\S]*?</svg>", flags=re.IGNORECASE)


def extract_svg(text):
    m = SVG_REGEX.search(text)
    return m.group(0).strip() if m else ""


def is_valid_svg(svg_text):
    if not svg_text:
        return False
    try:
        root = ET.fromstring(svg_text)
        return root.tag.endswith("svg")
    except ET.ParseError:
        return False


def fallback_svg(_prompt):
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        '<rect x="0" y="0" width="256" height="256" fill="white"/>'
        '<circle cx="128" cy="128" r="64" fill="black"/>'
        '</svg>'
    )


def generate_svg(prompt, max_new_tokens=512):
    chat_text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{prompt}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )
    inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            use_cache=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    svg = extract_svg(decoded)
    if not is_valid_svg(svg):
        svg = fallback_svg(prompt)
    return svg


test_prompt = "a simple blue bird icon"
pred_svg = generate_svg(test_prompt)
print(pred_svg[:500])
print("Valid SVG:", is_valid_svg(pred_svg))


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12

<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"><rect x="0" y="0" width="256" height="256" fill="white"/><circle cx="128" cy="128" r="64" fill="black"/></svg>
Valid SVG: True


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import time
import pandas as pd

# 1. Point the submission path directly to your mounted Google Drive
TEST_PROMPTS_PATH = "/content/test.csv" # Note: Make sure you actually uploaded test.csv to Colab!
SUBMISSION_PATH = "/content/drive/MyDrive/DL_Midterm_Overnighttt/submission.csv"

test_df = pd.read_csv(TEST_PROMPTS_PATH)

rows = []
invalid_count = 0
t0 = time.time()

print(f"Starting inference on {len(test_df)} rows. Saving to Drive...")

for idx, row in test_df.iterrows():
    # 2. Generate SVG using the safe, explicit function signature
    svg = generate_svg(row["prompt"])

    if not is_valid_svg(svg):
        invalid_count += 1
        svg = fallback_svg(row["prompt"])

    rows.append({"id": row["id"], "svg": svg})

    # Optional: Print progress every 100 rows so you know it hasn't frozen
    if (idx + 1) % 100 == 0:
        print(f"Processed {idx + 1}/{len(test_df)}...")

# 3. Save directly to Google Drive
sub_df = pd.DataFrame(rows)
sub_df.to_csv(SUBMISSION_PATH, index=False)

elapsed_min = (time.time() - t0) / 60
print(f"\n✅ Saved successfully to: {SUBMISSION_PATH}")
print(f"Rows processed: {len(sub_df)}")
print(f"Invalid/fallback count: {invalid_count}")
print(f"Runtime (minutes): {elapsed_min:.2f}")

sub_df.head()

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Starting inference on 1000 rows. Saving to Drive...


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=5

Processed 100/1000...


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

...

Processed 500/1000...


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Processed 600/1000...


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

KeyboardInterrupt: 

In [ ]:
print("Done")

In [ ]:
# Submission generation scaffold: expects Kaggle prompt file with columns `id,prompt`.
TEST_PROMPTS_PATH = "/content/test.csv"
SUBMISSION_PATH = "/kaggle/working/submission.csv"

test_df = pd.read_csv(TEST_PROMPTS_PATH)

rows = []
invalid_count = 0
t0 = time.time()

for _, row in test_df.iterrows():
    # Generate SVG using the globally defined model and tokenizer
    svg = generate_svg(row["prompt"])
    if not is_valid_svg(svg):
        invalid_count += 1
        svg = fallback_svg(row["prompt"])
    rows.append({"id": row["id"], "svg": svg})

sub_df = pd.DataFrame(rows)
sub_df.to_csv(SUBMISSION_PATH, index=False)

elapsed_min = (time.time() - t0) / 60
print(f"Saved: {SUBMISSION_PATH}")
print(f"Rows: {len(sub_df)}")
print(f"Invalid/fallback count: {invalid_count}")
print(f"Runtime (minutes): {elapsed_min:.2f}")
sub_df.head()


FileNotFoundError: [Errno 2] No such file or directory: '/content/test.csv'

In [ ]:
import os
import shutil
from google.colab import files

# Save the model and tokenizer
os.makedirs(CONFIG["output_dir"], exist_ok=True)
trainer.save_model(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])

print(f"Saved adapter + tokenizer to: {CONFIG['output_dir']}")
print("Zipping the model for Kaggle upload...")

# Compress the folder into a .zip file
zip_path = CONFIG["output_dir"]
shutil.make_archive(zip_path, 'zip', CONFIG["output_dir"])

# Trigger the download to your local machine
print("Starting download. Do not close this tab until it finishes!")
files.download(zip_path + '.zip')

Saved adapter + tokenizer to: /kaggle/working/qwen2b_svg_lora
Zipping the model for Kaggle upload...
Starting download. Do not close this tab until it finishes!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# import math
# import numpy as np

# def calculate_compactness(len_p, len_g):
#     """
#     Computes Compactness (C_i) - Weight: 0.03
#     len_p: character length of predicted SVG
#     len_g: character length of ground truth SVG
#     """
#     ratio = (len_p + 50) / (len_g + 50)
#     return math.exp(-abs(math.log(ratio)))

# def calculate_structural_similarity(ted):
#     """
#     Computes Structural Similarity (S_i) - Weight: 0.12
#     ted: Tree Edit Distance between the AST of predicted and ground truth SVGs
#     """
#     return math.exp(-ted / 25)

# def calculate_visual_fidelity(ssim, edge_f1):
#     """
#     Computes Visual Fidelity (V_i) - Weight: 0.85
#     ssim: Structural Similarity Index Measure of rendered images
#     edge_f1: Edge F1 score of rendered images
#     """
#     return (0.7 * ssim) + (0.3 * edge_f1)

# def evaluate_submission(predictions, ground_truths):
#     """
#     Calculates the final leaderboard score for a set of predictions.

#     predictions/ground_truths expected format per sample:
#     {
#         "valid": bool,       # Did it pass the validity gate?
#         "ssim": float,       # 0.0 to 1.0
#         "edge_f1": float,    # 0.0 to 1.0
#         "ted": float,        # Tree edit distance (integer >= 0)
#         "len_p": int,        # Length of generated SVG string
#         "len_g": int         # Length of reference SVG string
#     }
#     """
#     n_samples = len(predictions)
#     scores = []

#     for i in range(n_samples):
#         p = predictions[i]

#         # Stage 1: Hard Validity Gate
#         # If the SVG fails parsing, tag whitelist, rendering, or limits -> Score is 0
#         if not p["valid"]:
#             scores.append(0.0)
#             continue

#         # Stage 2: Quality Score Components
#         v_i = calculate_visual_fidelity(p["ssim"], p["edge_f1"])
#         s_i = calculate_structural_similarity(p["ted"])
#         c_i = calculate_compactness(p["len_p"], p["len_g"])

#         # Composite Score (Geometric Mean with weights)
#         # Qi = (Vi^0.85) * (Si^0.12) * (Ci^0.03)
#         q_i = (v_i ** 0.85) * (s_i ** 0.12) * (c_i ** 0.03)

#         # score_i = v_i * Q_i (where v_i is 1 since it passed the gate)
#         scores.append(q_i)

#     # Final Leaderboard Score is the mean of all row scores, scaled to 100
#     final_score = 100 * np.mean(scores)
#     return final_score

# # ==========================================
# # EXAMPLE USAGE
# # ==========================================
# if __name__ == "__main__":
#     # Mock data for two samples: one great, one invalid
#     mock_predictions = [
#         {
#             "valid": True,
#             "ssim": 0.92,
#             "edge_f1": 0.88,
#             "ted": 5,
#             "len_p": 1200,
#             "len_g": 1150
#         },
#         {
#             "valid": False, # e.g., exceeded 8,000 chars or had unclosed tag
#             "ssim": 0.0,
#             "edge_f1": 0.0,
#             "ted": 0,
#             "len_p": 8500,
#             "len_g": 1000
#         }
#     ]

#     score = evaluate_submission(mock_predictions, mock_predictions)
#     print(f"Estimated Kaggle Score: {score:.2f} / 100")

In [ ]:
# # ==============================================================================
# # 1. MOUNT GOOGLE DRIVE & SETUP FOLDERS
# # ==============================================================================
# from google.colab import drive
# drive.mount('/content/drive')

# import os
# import gc
# import re
# import xml.etree.ElementTree as ET
# import torch
# import pandas as pd
# import itertools
# from transformers import TrainingArguments
# from trl import SFTTrainer
# from unsloth import FastLanguageModel

# BASE_DIR = '/content/drive/MyDrive/DL_Midterm_Overnight'
# os.makedirs(BASE_DIR, exist_ok=True)

# # ==============================================================================
# # 2. DEFINE HELPER FUNCTIONS (Forces the correct versions into memory)
# # ==============================================================================
# SYSTEM_PROMPT = (
#     "You generate compact, valid SVG markup from user requests. "
#     "Return only SVG code with a single root <svg> element."
# )
# SVG_REGEX = re.compile(r"<svg[\s\S]*?</svg>", flags=re.IGNORECASE)

# def extract_svg(text):
#     m = SVG_REGEX.search(text)
#     return m.group(0).strip() if m else ""

# def is_valid_svg(svg_text):
#     if not svg_text:
#         return False
#     try:
#         root = ET.fromstring(svg_text)
#         return root.tag.endswith("svg")
#     except ET.ParseError:
#         return False

# def fallback_svg(_prompt):
#     return (
#         '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
#         '<rect x="0" y="0" width="256" height="256" fill="white"/>'
#         '<circle cx="128" cy="128" r="64" fill="black"/>'
#         '</svg>'
#     )

# def generate_svg(prompt, current_model, current_tokenizer, max_new_tokens=1024):
#     chat_text = (
#         "<|im_start|>system\n"
#         f"{SYSTEM_PROMPT}<|im_end|>\n"
#         "<|im_start|>user\n"
#         f"{prompt}<|im_end|>\n"
#         "<|im_start|>assistant\n"
#     )
#     inputs = current_tokenizer([chat_text], return_tensors="pt").to("cuda")

#     with torch.no_grad():
#         output_ids = current_model.generate(
#             input_ids=inputs.input_ids,
#             attention_mask=inputs.attention_mask,
#             max_new_tokens=max_new_tokens,
#             use_cache=False,
#             pad_token_id=current_tokenizer.eos_token_id,
#             do_sample=False, # Greedy decoding for stable XML
#         )

#     input_length = inputs.input_ids.shape[1]
#     decoded = current_tokenizer.decode(output_ids[0][input_length:], skip_special_tokens=True)
#     svg = extract_svg(decoded)
#     return svg if is_valid_svg(svg) else fallback_svg(prompt)


# # ==============================================================================
# # 3. OVERNIGHT GRID SEARCH LOOP (A100 Optimized)
# # ==============================================================================
# param_grid = {
#     "lora_r": [16, 32],
#     "lora_alpha": [16, 32],
#     "learning_rate": [1e-4, 2e-4, 3e-4]
# }

# keys, values = zip(*param_grid.items())
# configs_to_test = [dict(zip(keys, v)) for v in itertools.product(*values)]
# configs_to_test = [c for c in configs_to_test if c["lora_alpha"] >= c["lora_r"]]

# results = []

# for idx, run_config in enumerate(configs_to_test):
#     print(f"\n=========================================")
#     print(f"Starting Run {idx + 1}/{len(configs_to_test)}: {run_config}")
#     print(f"=========================================\n")

#     run_name = f"run_r{run_config['lora_r']}_a{run_config['lora_alpha']}_lr{run_config['learning_rate']}"
#     run_output_dir = f"{BASE_DIR}/{run_name}"

#     # Load Model fresh
#     model, tokenizer = FastLanguageModel.from_pretrained(
#         model_name="unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit",
#         max_seq_length=3072,
#         dtype=torch.bfloat16,
#         load_in_4bit=True,
#     )

#     model = FastLanguageModel.get_peft_model(
#         model,
#         r=run_config["lora_r"],
#         lora_alpha=run_config["lora_alpha"],
#         lora_dropout=0,
#         bias="none",
#         target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
#         use_gradient_checkpointing="unsloth",
#         random_state=42,
#     )

#     # Train
#     training_args = TrainingArguments(
#         output_dir=run_output_dir,
#         num_train_epochs=1,
#         per_device_train_batch_size=8,
#         gradient_accumulation_steps=2,
#         learning_rate=run_config["learning_rate"],
#         warmup_ratio=0.05,
#         weight_decay=0.01,
#         fp16=False,
#         bf16=True,
#         logging_steps=20,
#         save_strategy="no",
#         report_to="none",
#         optim="adamw_8bit",
#         lr_scheduler_type="cosine",
#         seed=42,
#     )

#     trainer = SFTTrainer(
#         model=model,
#         tokenizer=tokenizer,
#         train_dataset=train_text, # Assumes 'train_text' is already in memory from earlier steps
#         dataset_text_field="text",
#         max_seq_length=3072,
#         packing=True,
#         args=training_args,
#     )

#     train_result = trainer.train()

#     # Evaluate
#     FastLanguageModel.for_inference(model)
#     tokenizer.padding_side = "left"
#     model.eval()

#     valid_count = 0
#     test_prompts = ["a red circle", "a blue square", "a green tree icon", "a yellow star", "a simple house"]

#     for prompt in test_prompts:
#         svg_out = generate_svg(prompt, model, tokenizer)
#         if is_valid_svg(svg_out):
#             valid_count += 1

#     validity_rate = valid_count / len(test_prompts)
#     print(f"--> Validity Rate for {run_name}: {validity_rate * 100}%")

#     if validity_rate >= 0.8:
#         model.save_pretrained(run_output_dir)
#         tokenizer.save_pretrained(run_output_dir)

#     run_config["train_loss"] = train_result.metrics.get("train_loss", 999)
#     run_config["validity_rate"] = validity_rate
#     results.append(run_config)

#     pd.DataFrame(results).to_csv(f"{BASE_DIR}/overnight_search_results.csv", index=False)

#     # Clean up memory
#     del model, trainer, tokenizer
#     gc.collect()
#     torch.cuda.empty_cache()

# print(f"Overnight search complete! Results saved to {BASE_DIR}")

In [ ]:
# # ==============================================================================
# # 1. MOUNT GOOGLE DRIVE & SETUP FOLDERS
# # ==============================================================================
# from google.colab import drive
# drive.mount('/content/drive')

# import os
# import gc
# import re
# import xml.etree.ElementTree as ET
# import torch
# import pandas as pd
# import itertools
# from transformers import TrainingArguments
# from trl import SFTTrainer
# from unsloth import FastLanguageModel

# BASE_DIR = '/content/drive/MyDrive/DL_Midterm_Overnight_v2'
# os.makedirs(BASE_DIR, exist_ok=True)

# # ==============================================================================
# # 2. DEFINE HELPER FUNCTIONS (Forces the correct versions into memory)
# # ==============================================================================
# SYSTEM_PROMPT = (
#     "You generate compact, valid SVG markup from user requests. "
#     "Return only SVG code with a single root <svg> element."
# )
# SVG_REGEX = re.compile(r"<svg[\s\S]*?</svg>", flags=re.IGNORECASE)

# def extract_svg(text):
#     m = SVG_REGEX.search(text)
#     return m.group(0).strip() if m else ""

# def is_valid_svg(svg_text):
#     if not svg_text:
#         return False
#     try:
#         root = ET.fromstring(svg_text)
#         return root.tag.endswith("svg")
#     except ET.ParseError:
#         return False

# def fallback_svg(_prompt):
#     return (
#         '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
#         '<rect x="0" y="0" width="256" height="256" fill="white"/>'
#         '<circle cx="128" cy="128" r="64" fill="black"/>'
#         '</svg>'
#     )

# def generate_svg(prompt, current_model, current_tokenizer, max_new_tokens=1024):
#     chat_text = (
#         "<|im_start|>system\n"
#         f"{SYSTEM_PROMPT}<|im_end|>\n"
#         "<|im_start|>user\n"
#         f"{prompt}<|im_end|>\n"
#         "<|im_start|>assistant\n"
#     )
#     inputs = current_tokenizer([chat_text], return_tensors="pt").to("cuda")

#     with torch.no_grad():
#         output_ids = current_model.generate(
#             input_ids=inputs.input_ids,
#             attention_mask=inputs.attention_mask,
#             max_new_tokens=max_new_tokens,
#             use_cache=False,
#             pad_token_id=current_tokenizer.eos_token_id,
#             do_sample=False, # Greedy decoding for stable XML
#         )

#     input_length = inputs.input_ids.shape[1]
#     decoded = current_tokenizer.decode(output_ids[0][input_length:], skip_special_tokens=True)
#     svg = extract_svg(decoded)
#     return svg if is_valid_svg(svg) else fallback_svg(prompt)


# # ==============================================================================
# # 3. OVERNIGHT GRID SEARCH LOOP (A100 Optimized)
# # ==============================================================================
# param_grid = {
#     "lora_r": [16, 32, 64],               # 64 adds massive capacity for complex XML
#     "alpha_multiplier": [1, 2],           # Alpha will be r * 1 or r * 2
#     "learning_rate": [1e-4, 2e-4, 3e-4],
#     "warmup_ratio": [0.05, 0.1]           # 5% vs 10% warmup steps
# }

# # Generate all combinations
# keys, values = zip(*param_grid.items())
# raw_configs = [dict(zip(keys, v)) for v in itertools.product(*values)]

# configs_to_test = []
# for c in raw_configs:
#     # Calculate the actual lora_alpha dynamically
#     actual_alpha = c["lora_r"] * c["alpha_multiplier"]

#     configs_to_test.append({
#         "lora_r": c["lora_r"],
#         "lora_alpha": actual_alpha,
#         "learning_rate": c["learning_rate"],
#         "warmup_ratio": c["warmup_ratio"]
#     })

# print(f"Total configurations to test: {len(configs_to_test)}")
# # (3 R's) * (2 Multipliers) * (3 LRs) * (2 Warmups) = 36 Total Runs
# # On an A100, this should take roughly 8 to 12 hours.

# results = []

# for idx, run_config in enumerate(configs_to_test):
#     print(f"\n=========================================")
#     print(f"Starting Run {idx + 1}/{len(configs_to_test)}: {run_config}")
#     print(f"=========================================\n")

#     run_name = f"run_r{run_config['lora_r']}_a{run_config['lora_alpha']}_lr{run_config['learning_rate']}_wr{run_config['warmup_ratio']}"
#     run_output_dir = f"{BASE_DIR}/{run_name}"

#     # Load Model fresh
#     model, tokenizer = FastLanguageModel.from_pretrained(
#         model_name="unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit",
#         max_seq_length=3072,
#         dtype=torch.bfloat16,
#         load_in_4bit=True,
#     )

#     model = FastLanguageModel.get_peft_model(
#         model,
#         r=run_config["lora_r"],
#         lora_alpha=run_config["lora_alpha"],
#         lora_dropout=0,
#         bias="none",
#         target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
#         use_gradient_checkpointing="unsloth",
#         random_state=42,
#     )

#     # Train
#     training_args = TrainingArguments(
#         output_dir=run_output_dir,
#         num_train_epochs=1,
#         per_device_train_batch_size=8,
#         gradient_accumulation_steps=2,
#         learning_rate=run_config["learning_rate"],
#         warmup_ratio=0.05,
#         weight_decay=0.01,
#         fp16=False,
#         bf16=True,
#         logging_steps=20,
#         save_strategy="no",
#         report_to="none",
#         optim="adamw_8bit",
#         lr_scheduler_type="cosine",
#         seed=42,
#     )

#     trainer = SFTTrainer(
#         model=model,
#         tokenizer=tokenizer,
#         train_dataset=train_text, # Assumes 'train_text' is already in memory from earlier steps
#         dataset_text_field="text",
#         max_seq_length=3072,
#         packing=True,
#         args=training_args,
#     )

#     train_result = trainer.train()

#     # Evaluate
#     FastLanguageModel.for_inference(model)
#     tokenizer.padding_side = "left"
#     model.eval()

#     valid_count = 0
#     test_prompts = ["a red circle", "a blue square", "a green tree icon", "a yellow star", "a simple house"]

#     for prompt in test_prompts:
#         svg_out = generate_svg(prompt, model, tokenizer)
#         if is_valid_svg(svg_out):
#             valid_count += 1

#     validity_rate = valid_count / len(test_prompts)
#     print(f"--> Validity Rate for {run_name}: {validity_rate * 100}%")

#     if validity_rate >= 0.8:
#         model.save_pretrained(run_output_dir)
#         tokenizer.save_pretrained(run_output_dir)

#     run_config["train_loss"] = train_result.metrics.get("train_loss", 999)
#     run_config["validity_rate"] = validity_rate
#     results.append(run_config)

#     pd.DataFrame(results).to_csv(f"{BASE_DIR}/overnight_search_results.csv", index=False)

#     # Clean up memory
#     del model, trainer, tokenizer
#     gc.collect()
#     torch.cuda.empty_cache()

# print(f"Overnight search complete! Results saved to {BASE_DIR}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import time
import re
import pandas as pd
import torch
import xml.etree.ElementTree as ET
from unsloth import FastLanguageModel

# ==========================================
# 1. PATHS (Update MODEL_PATH to your saved run)
# ==========================================
# Look in your Google Drive and paste the exact path to your best saved model folder here:
MODEL_PATH = "/content/drive/MyDrive/DL_Midterm_Overnight_v2/qwen2b_svg_lora"
TEST_PROMPTS_PATH = "/content/test.csv"
SUBMISSION_PATH = "/content/drive/MyDrive/DL_Midterm_Overnight/submission.csv"

# ==========================================
# 2. LOAD THE MODEL & APPLY INFERENCE FIXES
# ==========================================
print(f"Loading model from {MODEL_PATH}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH, # Unsloth automatically loads the base model and attaches your LoRA weights!
    max_seq_length=3072,
    dtype=torch.bfloat16,  # Keep this if you are still on the A100/H100
    load_in_4bit=True,
)
# Apply the Magic Bullet stability fixes
FastLanguageModel.for_inference(model)
tokenizer.padding_side = "left"
model.eval()

# ==========================================
# 3. HELPER FUNCTIONS & SCORE BOOSTERS
# ==========================================
SYSTEM_PROMPT = (
    "You generate compact, valid SVG markup from user requests. "
    "Return only SVG code with a single root <svg> element."
)
SVG_REGEX = re.compile(r"<svg[\s\S]*?</svg>", flags=re.IGNORECASE)

def extract_svg(text):
    m = SVG_REGEX.search(text)
    return m.group(0).strip() if m else ""

def is_valid_svg(svg_text):
    if not svg_text: return False
    try:
        root = ET.fromstring(svg_text)
        return root.tag.endswith("svg")
    except ET.ParseError:
        return False

def fallback_svg(_prompt):
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        '<rect x="0" y="0" width="256" height="256" fill="white"/>'
        '<circle cx="128" cy="128" r="64" fill="black"/>'
        '</svg>'
    )

def optimize_svg(svg_string):
    """Applies minification and rounding to boost the Compactness (Ci) Score"""
    # 1. Minify (remove newlines, tabs, extra spaces)
    svg = svg_string.replace('\n', '').replace('\r', '').replace('\t', '')
    svg = re.sub(r'>\s+<', '><', svg)
    svg = re.sub(r'\s*=\s*', '=', svg)
    svg = re.sub(r'\s+', ' ', svg)

    # 2. Round coordinates to 1 decimal place to save characters
    def round_match(match):
        val = float(match.group(0))
        return str(int(val)) if val.is_integer() else f"{val:.1f}"
    svg = re.sub(r'-?\d+\.\d+', round_match, svg)

    return svg.strip()

# Make sure these two lines are set for batched inference to work!
tokenizer.padding_side = "left"
tokenizer.pad_token_id = tokenizer.eos_token_id

BATCH_SIZE = 64 # Safe for Kaggle's T4 GPUs. If it throws an OOM error, drop to 8.
torch.backends.cudnn.benchmark = True
def generate_svg_batch(prompts):
    # 1. Format the entire batch of prompts
    chat_texts = []
    for prompt in prompts:
        chat_text = (
            "<|im_start|>system\n"
            f"{SYSTEM_PROMPT}<|im_end|>\n"
            "<|im_start|>user\n"
            f"{prompt}<|im_end|>\n"
            "<|im_start|>assistant\n"
        )
        chat_texts.append(chat_text)

    # 2. Tokenize them all together with padding
    inputs = tokenizer(chat_texts, return_tensors="pt", padding=True).to("cuda")

    # 3. Generate all 16 SVGs simultaneously
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            max_length=None,
            use_cache=False,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=False, # Greedy decoding remains on for stability
        )

    # 4. Decode the batch
    input_length = inputs.input_ids.shape[1]
    decoded_batch = tokenizer.batch_decode(output_ids[:, input_length:], skip_special_tokens=True)

    # 5. Extract and optimize
    results = []
    for i, decoded in enumerate(decoded_batch):
        svg = extract_svg(decoded)
        if is_valid_svg(svg):
            results.append(optimize_svg(svg))
        else:
            results.append(fallback_svg(prompts[i]))

    return results

# ==========================================
# RUN BATCHED INFERENCE LOOP
# ==========================================
print(f"Reading test prompts from {TEST_PROMPTS_PATH}...")
test_df = pd.read_csv(TEST_PROMPTS_PATH)

rows = []
invalid_count = 0
t0 = time.time()

print(f"Starting BATCHED inference on {len(test_df)} rows. Saving to {SUBMISSION_PATH}...")

# Process the dataframe in chunks of BATCH_SIZE
for i in range(0, len(test_df), BATCH_SIZE):
    batch_df = test_df.iloc[i : i + BATCH_SIZE]
    batch_prompts = batch_df["prompt"].tolist()
    batch_ids = batch_df["id"].tolist()

    # Send the batch to the GPU
    batch_svgs = generate_svg_batch(batch_prompts)

    # Log the results
    for j, svg in enumerate(batch_svgs):
        if svg.startswith('<svg') and 'fill="white"' in svg and 'circle cx="128"' in svg:
            invalid_count += 1
        print("id", batch_ids[j], "svg", svg)
        rows.append({"id": batch_ids[j], "svg": svg})

    # Print progress
    if (i + len(batch_prompts)) % (BATCH_SIZE * 10) == 0 or (i + len(batch_prompts)) == len(test_df):
        elapsed = (time.time() - t0) / 60
        print(f"Processed {i + len(batch_prompts)}/{len(test_df)}... (Elapsed: {elapsed:.2f} min)")

# Save
os.makedirs(os.path.dirname(SUBMISSION_PATH), exist_ok=True)
sub_df = pd.DataFrame(rows)
sub_df.to_csv(SUBMISSION_PATH, index=False)

final_time = (time.time() - t0) / 60
print(f"\n✅ Saved successfully to: {SUBMISSION_PATH}")
print(f"Fallback count: {invalid_count}")
print(f"Total Batched Runtime (minutes): {final_time:.2f}")

Loading model from /content/drive/MyDrive/DL_Midterm_Overnight_v2/qwen2b_svg_lora...
==((====))==  Unsloth 2026.3.15: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.179 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Reading test prompts from /content/test.csv...
Starting BATCHED inference on 1000 rows. Saving to /content/drive/MyDrive/DL_Midterm_Overnight/submission.csv...


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

KeyboardInterrupt: 

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import time
import re
import gc
import pandas as pd
import torch
import xml.etree.ElementTree as ET
from unsloth import FastLanguageModel
from vllm import LLM, SamplingParams

# ==========================================
# 1. PATHS
# ==========================================
MODEL_PATH = "/content/drive/MyDrive/DL_Midterm_Overnight_v2/qwen2b_svg_lora"
MERGED_PATH = "/content/merged_qwen_svg"  # Temporary local storage for the merged model
TEST_PROMPTS_PATH = "/content/test.csv"
SUBMISSION_PATH = "/content/drive/MyDrive/DL_Midterm_Overnight/submission-v4.csv"

# ==========================================
# 2. HELPER FUNCTIONS & SCORE BOOSTERS
# ==========================================
# SYSTEM_PROMPT = (
#     "You generate compact, valid SVG markup from user requests. "
#     "Return only SVG code with a single root <svg> element."
# )
SVG_REGEX = re.compile(r"<svg[\s\S]*?</svg>", flags=re.IGNORECASE)

def extract_svg(text):
    m = SVG_REGEX.search(text)
    if m:
        svg = m.group(0).strip()
        if not svg.endswith("</svg>"):
            svg += "</svg>"
        return svg
    return ""

def is_valid_svg(svg_text):
    if not svg_text: return False
    try:
        return ET.fromstring(svg_text).tag.endswith("svg")
    except ET.ParseError:
        return False

def fallback_svg(_prompt):
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        '<rect x="0" y="0" width="256" height="256" fill="white"/>'
        '<circle cx="128" cy="128" r="64" fill="black"/>'
        '</svg>'
    )

def optimize_svg(svg_string):
    svg = svg_string.replace('\n', '').replace('\r', '').replace('\t', '')
    svg = re.sub(r'>\s+<', '><', svg)
    svg = re.sub(r'\s*=\s*', '=', svg)
    svg = re.sub(r'\s+', ' ', svg)
    def round_match(match):
        val = float(match.group(0))
        return str(int(val)) if val.is_integer() else f"{val:.1f}"
    return re.sub(r'-?\d+\.\d+', round_match, svg).strip()


# ==========================================
# 3. PREPARE THE MODEL FOR vLLM (Merge LoRA)
# ==========================================
if not os.path.exists(MERGED_PATH):
    print(f"Loading Unsloth model from {MODEL_PATH} to merge weights...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_PATH,
        max_seq_length=3072,
        dtype=torch.bfloat16,
        load_in_4bit=False, # Must be False so we can export a clean 16-bit model for vLLM
    )

    print(f"Saving merged 16-bit model to {MERGED_PATH}...")
    model.save_pretrained_merged(MERGED_PATH, tokenizer, save_method="merged_16bit")

    # CRITICAL: We must free the GPU memory so vLLM can claim it
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print("Merge complete. GPU memory cleared.")
else:
    print(f"Merged model already found at {MERGED_PATH}. Skipping merge.")


# ==========================================
# 4. START vLLM ENGINE
# ==========================================
print("\nBooting up vLLM Engine...")
# vLLM automatically allocates GPU memory optimally
llm = LLM(model=MERGED_PATH, max_model_len=3072, tensor_parallel_size=1)

# We set strictly greedy decoding (temperature=0.0) for stable XML
# NEW CODE
sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=2500,
    stop=["</svg>", "<|im_end|>", "<|endoftext|>"],  # <-- Add </svg>
    repetition_penalty=1.2,  # <-- Try higher
    # top_p=0.9,  # Optionally add
)

SYSTEM_PROMPT = (
    "You generate compact, valid SVG markup from user requests. "
    "Return only SVG code with a single root <svg> element. "
    "Always end your response with a single closing </svg> tag. "
    "Do not repeat elements. Do not generate extra text after </svg>."
)
# ==========================================
# 5. RUN MASSIVE PARALLEL INFERENCE
# ==========================================
print(f"Reading test prompts from {TEST_PROMPTS_PATH}...")
test_df = pd.read_csv(TEST_PROMPTS_PATH)
prompts = test_df["prompt"].tolist()
ids = test_df["id"].tolist()

# Format all prompts into the ChatML format ahead of time
chat_texts = []
for p in prompts:
    chat_text = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{p}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    chat_texts.append(chat_text)

print(f"Feeding {len(chat_texts)} prompts to vLLM. Hold on tight...")
t0 = time.time()

# BOOM. vLLM processes the entire array using continuous batching.
outputs = llm.generate(chat_texts, sampling_params)

# Process the outputs and apply fallbacks
rows = []
invalid_count = 0

for i, output in enumerate(outputs):
    # output.outputs[0].text contains the generated string
    generated_text = output.outputs[0].text
    svg = extract_svg(generated_text)

    if is_valid_svg(svg):
        final_svg = optimize_svg(svg)
    else:
        invalid_count += 1
        final_svg = fallback_svg(prompts[i])

    rows.append({"id": ids[i], "svg": final_svg})

# Save Results
os.makedirs(os.path.dirname(SUBMISSION_PATH), exist_ok=True)
pd.DataFrame(rows).to_csv(SUBMISSION_PATH, index=False)

final_time = (time.time() - t0) / 60
print(f"\n✅ Blazing fast inference complete! Saved to {SUBMISSION_PATH}")
print(f"Fallback count: {invalid_count}")
print(f"Total Batched Runtime (minutes): {final_time:.2f}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading Unsloth model from /content/drive/MyDrive/DL_Midterm_Overnight_v2/qwen2b_svg_lora to merge weights...
==((====))==  Unsloth 2026.3.15: Fast Qwen2 patching. Transformers: 4.57.6. vLLM: 0.18.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.179 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/265 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:12<00:00, 12.11s/it]


Unsloth: Merge process complete. Saved to `/content/merged_qwen_svg`
Merge complete. GPU memory cleared.

Booting up vLLM Engine...
INFO 03-26 23:10:16 [utils.py:233] non-default args: {'max_model_len': 3072, 'disable_log_stats': True, 'model': '/content/merged_qwen_svg'}
INFO 03-26 23:10:29 [model.py:533] Resolved architecture: Qwen2ForCausalLM
INFO 03-26 23:10:29 [model.py:1582] Using max model len 3072
INFO 03-26 23:10:29 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 03-26 23:10:29 [vllm.py:754] Asynchronous scheduling is enabled.
WARNING 03-26 23:10:30 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
INFO 03-26 23:11:11 [llm.py:391] Supported tasks: ['generate']
Reading test prompts from /content/test.csv...
Feeding 1000 prompts to

Rendering prompts:   0%|          | 0/1000 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1000 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…


✅ Blazing fast inference complete! Saved to /content/drive/MyDrive/DL_Midterm_Overnight/submission-v4.csv
Fallback count: 1000
Total Batched Runtime (minutes): 2.02


## Notes

- Keep a fixed seed, runtime logs, and invalid-generation counts (required by `contest_docs/05`).
- If you use Kaggle-packaged datasets (`svg-train-public`, `svg-test-public-prompts`, `svg-utils`), swap paths into the loading cells.
- For stricter alignment with Unsloth templates, copy the latest prompt-formatting snippets from the official Qwen3.5-2B notebook linked above.

In [4]:
# OLD CODE
test_df = pd.read_csv(SUBMISSION_PATH)

In [5]:
test_df.head(20)

,id,svg
0,fa1d8fa7-080f-4269-a9cf-a17562c9a0ca,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
1,6eede943219547c22ac56085027d33cc,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
2,ea045c7a247166f061ce504d9b7ccaab,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
3,8fe82f3af89e487b31236ca829c3f071,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
4,600464e4d92c75338462271a09b3f176,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
5,9e831fb6831745f4d15156b7a95e4f92,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
6,8cbefcd53fd5dfa2cbe1b374356ed709,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
7,625181a2-600d-4db5-83c6-1a34676eb0dc,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
8,8ba1bd7c-211c-43b0-a4aa-0e6e33c92486,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
9,4e56e508270eb0d02291927638d3c685,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
